<a href="https://colab.research.google.com/github/sarwathasan72-svg/Masai_AIML_Capstone/blob/main/IITR_Capstone_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""Synthetic shipment data for the Delay Intelligence capstone.
Run once:  python generate_data.py   (writes data/hubs.csv, data/raw_shipments.csv, data/new_shipments.csv)
Do not edit any constant: every number in the brief was measured on this exact output."""
import os
import numpy as np
import pandas as pd

HUBS = [
    ("HUB01", "Mumbai", "West", 1, 1500), ("HUB02", "Delhi", "North", 1, 1600),
    ("HUB03", "Bengaluru", "South", 1, 1450), ("HUB04", "Hyderabad", "South", 1, 1350),
    ("HUB05", "Chennai", "South", 1, 1300), ("HUB06", "Pune", "West", 1, 1250),
    ("HUB07", "Kolkata", "East", 1, 1200), ("HUB08", "Ahmedabad", "West", 1, 1150),
    ("HUB09", "Jaipur", "North", 2, 950), ("HUB10", "Lucknow", "North", 2, 900),
    ("HUB11", "Chandigarh", "North", 2, 850), ("HUB12", "Indore", "West", 2, 820),
    ("HUB13", "Nagpur", "West", 2, 780), ("HUB14", "Coimbatore", "South", 2, 740),
    ("HUB15", "Kochi", "South", 2, 700), ("HUB16", "Surat", "West", 2, 650),
    ("HUB17", "Patna", "East", 3, 520), ("HUB18", "Ranchi", "East", 3, 480),
    ("HUB19", "Guwahati", "East", 3, 450), ("HUB20", "Raipur", "East", 3, 420),
    ("HUB21", "Bhubaneswar", "East", 3, 390), ("HUB22", "Dehradun", "North", 3, 350),
    ("HUB23", "Siliguri", "East", 3, 310), ("HUB24", "Agartala", "East", 3, 270),
]
DIST_MU = [180, 320, 480]      # mean lane distance (km) for hub tier 1 / 2 / 3
DIST_SD = [50, 90, 150]
STOPS_MU = [55, 70, 85]        # mean stops on the rider route for hub tier 1 / 2 / 3
HUB_EFFECT = [0, 2, 4]         # extra hours for hub tier 1 / 2 / 3
# How a hub really operates (1 = metro-like, 2 = regional, 3 = stretched). Four hubs operate unlike their tier label.
OPERATING_TIER = {"HUB07": 2, "HUB13": 3, "HUB15": 3, "HUB22": 2}
BASE_HOURS = {"Express": 10, "Standard": 28, "Economy": 58}


def simulate(n_per_hub, seed):
    """Clean shipments, n_per_hub rows for each of the 24 hubs, in hub order."""
    np.random.seed(seed)
    hubs = pd.DataFrame(HUBS, columns=["hub_id", "hub_city", "region", "hub_tier", "hub_daily_capacity"])
    n = n_per_hub * len(hubs)
    op_tier = [OPERATING_TIER.get(h, t) for h, t in zip(hubs["hub_id"], hubs["hub_tier"])]
    tier = np.repeat(op_tier, n_per_hub) - 1
    hub_id = np.repeat(hubs["hub_id"].values, n_per_hub)
    distance = np.clip(np.random.normal(np.array(DIST_MU)[tier], np.array(DIST_SD)[tier]), 15, 1500).round(1)
    stops = np.clip(np.random.normal(np.array(STOPS_MU)[tier], 10), 20, 140).round().astype(int)
    weight = np.clip(np.random.lognormal(0.6, 0.7, n), 0.1, 40).round(2)
    value = np.random.lognormal(7.0, 0.8, n).round().astype(int)
    service = np.random.choice(["Express", "Standard", "Economy"], n, p=[0.2, 0.55, 0.25])
    pay = np.random.choice(["Prepaid", "COD"], n, p=[0.6, 0.4])
    rain = (np.random.rand(n) < 0.18).astype(int)
    day = np.random.randint(0, 90, n)
    hour = np.random.randint(6, 22, n)
    minute = np.random.randint(0, 60, n)
    ts = pd.Timestamp("2026-07-01") + pd.to_timedelta(day, unit="D") + pd.to_timedelta(hour * 60 + minute, unit="m")
    sale = ((ts >= pd.Timestamp("2026-09-15")) & (ts < pd.Timestamp("2026-09-29"))).astype(int)
    weekend = (ts.dayofweek >= 5).astype(int)
    hours = (np.array([BASE_HOURS[s] for s in service]) + 0.03 * distance + 0.10 * stops
             + np.array(HUB_EFFECT)[tier] + 0.12 * weight + 4 * rain + 7 * sale
             + 3 * (pay == "COD") + 2 * weekend + 3 * (hour >= 18)
             + 12 * sale * (service == "Express") + 8 * (distance > 400) * rain
             + np.random.normal(0, 4, n)).round(2)
    return pd.DataFrame({
        "hub_id": hub_id, "service_type": service, "payment_mode": pay,
        "parcel_weight_kg": weight, "distance_km": distance, "declared_value": value,
        "n_stops": stops, "rain_flag": rain, "sale_event_flag": sale, "dispatch_time": ts,
        "label_print_batch": np.random.randint(1000, 9999, n),
        "scanner_build_no": np.random.randint(1, 40, n),
        "tape_roll_no": np.random.randint(1, 500, n),
        "sticker_serial_mod": np.random.uniform(0, 1, n).round(3),
        "shelf_slot_no": np.random.randint(1, 300, n),
        "delivery_hours": hours,
    })


def make_messy(df, seed, n_missing, n_case, n_sentinel, n_dupes, id_start):
    """Turn a clean table into a raw export: text columns, gaps, duplicates, a sentinel."""
    np.random.seed(seed)
    df = df.sample(frac=1.0, random_state=seed).reset_index(drop=True)
    n = len(df)
    df.insert(0, "shipment_id", ["SHP" + str(id_start + i) for i in range(n)])
    df["dispatch_time"] = df["dispatch_time"].dt.strftime("%Y-%m-%d %H:%M")
    df["declared_value"] = df["declared_value"].map(lambda v: "\u20b9" + format(v, ","))
    for col, k in [("distance_km", n_missing[0]), ("parcel_weight_kg", n_missing[1]),
                   ("declared_value", n_missing[2]), ("payment_mode", n_missing[3])]:
        df.loc[np.random.choice(n, k, replace=False), col] = np.nan
    pick = np.random.choice(n, n_case, replace=False)
    shapes = np.random.randint(0, 3, n_case)
    for i, s in zip(pick, shapes):
        v = df.at[i, "service_type"]
        df.at[i, "service_type"] = [v.upper(), v.lower(), "  " + v + " "][s]
    if n_sentinel:
        df.loc[np.random.choice(n, n_sentinel, replace=False), "parcel_weight_kg"] = 999.0
    if n_dupes:
        df = pd.concat([df, df.iloc[np.random.choice(n, n_dupes, replace=False)]], ignore_index=True)
    return df


if __name__ == "__main__":
    os.makedirs("data", exist_ok=True)
    hubs = pd.DataFrame(HUBS, columns=["hub_id", "hub_city", "region", "hub_tier", "hub_daily_capacity"])
    hubs.to_csv("data/hubs.csv", index=False)
    raw = make_messy(simulate(50, 2604), seed=7, n_missing=(36, 24, 12, 18),
                     n_case=120, n_sentinel=5, n_dupes=24, id_start=10001)
    raw.to_csv("data/raw_shipments.csv", index=False)
    new = make_messy(simulate(2, 99), seed=8, n_missing=(3, 2, 1, 2),
                     n_case=6, n_sentinel=0, n_dupes=0, id_start=90001)
    new = new.drop(columns=["delivery_hours"])
    new.to_csv("data/new_shipments.csv", index=False)
    print(len(hubs), len(raw), len(new))

24 1224 48


In [ ]:
import numpy as np
import pandas as pd

In [ ]:
hubs_df = pd.read_csv("/content/data/hubs.csv")
hubs_df.head()

,hub_id,hub_city,region,hub_tier,hub_daily_capacity
0,HUB01,Mumbai,West,1,1500
1,HUB02,Delhi,North,1,1600
2,HUB03,Bengaluru,South,1,1450
3,HUB04,Hyderabad,South,1,1350
4,HUB05,Chennai,South,1,1300


In [ ]:
hubs_df.shape

(24, 5)

In [ ]:
raw_ship_df = pd.read_csv("/content/data/raw_shipments.csv")
raw_ship_df.head()

,shipment_id,hub_id,service_type,payment_mode,parcel_weight_kg,distance_km,declared_value,n_stops,rain_flag,sale_event_flag,dispatch_time,label_print_batch,scanner_build_no,tape_roll_no,sticker_serial_mod,shelf_slot_no,delivery_hours
0,SHP10001,HUB10,express,COD,0.62,398.3,"₹1,290",62,0,0,2026-08-23 19:40,4594,25,167,0.650,13,34.79
1,SHP10002,HUB10,Standard,COD,4.25,312.2,"₹1,030",80,0,0,2026-08-16 07:42,4066,31,169,0.045,243,58.56
2,SHP10003,HUB21,Economy,Prepaid,0.81,421.3,₹441,73,0,0,2026-08-01 12:15,3090,25,170,0.915,80,76.56
3,SHP10004,HUB03,Standard,Prepaid,4.43,240.7,"₹3,122",42,0,0,2026-07-20 17:44,1796,21,338,0.380,175,41.18
4,SHP10005,HUB14,Standard,Prepaid,0.91,317.9,₹276,72,0,0,2026-07-19 17:31,4231,34,215,0.824,162,47.32


In [ ]:
new_ship_df = pd.read_csv("/content/data/new_shipments.csv")
new_ship_df.head()

,shipment_id,hub_id,service_type,payment_mode,parcel_weight_kg,distance_km,declared_value,n_stops,rain_flag,sale_event_flag,dispatch_time,label_print_batch,scanner_build_no,tape_roll_no,sticker_serial_mod,shelf_slot_no
0,SHP90001,HUB22,Standard,Prepaid,1.52,478.7,₹483,78,0,0,2026-08-03 17:26,9034,5,311,0.710,273
1,SHP90002,HUB24,Standard,COD,0.76,405.5,"₹1,137",94,1,0,2026-08-21 12:52,7336,8,163,0.784,284
2,SHP90003,HUB18,Standard,Prepaid,2.00,552.4,₹195,99,0,0,2026-08-07 16:29,9987,3,291,0.927,140
3,SHP90004,HUB12,Economy,Prepaid,1.66,243.0,"₹1,018",65,0,0,2026-08-28 14:41,4749,14,257,0.405,21
4,SHP90005,HUB22,Express,COD,0.72,219.1,NaN,72,0,0,2026-08-25 18:11,1364,11,442,0.832,164


In [ ]:
PROMISED_HOURS = {"Express": 36, "Standard": 60, "Economy": 96}
FEATURES = [
 "promised_hours", "distance_km", "parcel_weight_kg", "log_declared_value",
"n_stops",
 "rain_flag", "sale_event_flag", "is_cod", "is_weekend", "dispatch_hour",
 "hub_tier", "hub_daily_capacity",
 "label_print_batch", "scanner_build_no", "tape_roll_no", "sticker_serial_mod",
"shelf_slot_no",
]
NOISE_COLUMNS = ["label_print_batch", "scanner_build_no", "tape_roll_no",
"sticker_serial_mod", "shelf_slot_no"]

In [ ]:
PROMISED_HOURS

{'Express': 36, 'Standard': 60, 'Economy': 96}

In [ ]:
FEATURES

['promised_hours',
 'distance_km',
 'parcel_weight_kg',
 'log_declared_value',
 'n_stops',
 'rain_flag',
 'sale_event_flag',
 'is_cod',
 'is_weekend',
 'dispatch_hour',
 'hub_tier',
 'hub_daily_capacity',
 'label_print_batch',
 'scanner_build_no',
 'tape_roll_no',
 'sticker_serial_mod',
 'shelf_slot_no']

In [ ]:
len(FEATURES)

17

In [ ]:
raw_ship_df.head()

,shipment_id,hub_id,service_type,payment_mode,parcel_weight_kg,distance_km,declared_value,n_stops,rain_flag,sale_event_flag,dispatch_time,label_print_batch,scanner_build_no,tape_roll_no,sticker_serial_mod,shelf_slot_no,delivery_hours
0,SHP10001,HUB10,express,COD,0.62,398.3,"₹1,290",62,0,0,2026-08-23 19:40,4594,25,167,0.650,13,34.79
1,SHP10002,HUB10,Standard,COD,4.25,312.2,"₹1,030",80,0,0,2026-08-16 07:42,4066,31,169,0.045,243,58.56
2,SHP10003,HUB21,Economy,Prepaid,0.81,421.3,₹441,73,0,0,2026-08-01 12:15,3090,25,170,0.915,80,76.56
3,SHP10004,HUB03,Standard,Prepaid,4.43,240.7,"₹3,122",42,0,0,2026-07-20 17:44,1796,21,338,0.380,175,41.18
4,SHP10005,HUB14,Standard,Prepaid,0.91,317.9,₹276,72,0,0,2026-07-19 17:31,4231,34,215,0.824,162,47.32


In [ ]:
raw_ship_df.duplicated()

,0
0,False
1,False
2,False
3,False
4,False
...,...
1219,True
1220,True
1221,True
1222,True


In [ ]:
raw_ship_df.duplicated().sum()

np.int64(24)

In [ ]:
raw_ship_df.duplicated().sum()*100/len(raw_ship_df)

np.float64(1.9607843137254901)

In [ ]:
len(raw_ship_df)

1224

In [ ]:
24*100/1224

1.9607843137254901

In [ ]:
raw_ship_df.duplicated().mean()*100

np.float64(1.9607843137254901)

In [ ]:
raw_ship_df.sample(5, random_state = 42)

,shipment_id,hub_id,service_type,payment_mode,parcel_weight_kg,distance_km,declared_value,n_stops,rain_flag,sale_event_flag,dispatch_time,label_print_batch,scanner_build_no,tape_roll_no,sticker_serial_mod,shelf_slot_no,delivery_hours
661,SHP10662,HUB16,Standard,Prepaid,7.81,316.6,"₹1,536",80,0,1,2026-09-25 09:48,2021,12,416,0.006,173,55.16
220,SHP10221,HUB08,Economy,COD,1.94,134.7,"₹1,246",66,1,0,2026-08-19 19:40,9694,15,451,0.627,21,76.38
155,SHP10156,HUB18,standard,Prepaid,2.66,541.0,"₹1,108",80,0,0,2026-07-07 15:02,2947,22,253,0.239,124,54.75
677,SHP10678,HUB03,Standard,Prepaid,0.71,255.5,₹560,58,0,0,2026-08-14 19:59,7558,33,227,0.884,203,44.17
911,SHP10912,HUB04,Standard,COD,3.75,167.5,₹618,66,0,0,2026-08-14 12:38,3127,10,10,0.626,287,44.30


In [ ]:
cols = raw_ship_df.columns.tolist()
cols

['shipment_id',
 'hub_id',
 'service_type',
 'payment_mode',
 'parcel_weight_kg',
 'distance_km',
 'declared_value',
 'n_stops',
 'rain_flag',
 'sale_event_flag',
 'dispatch_time',
 'label_print_batch',
 'scanner_build_no',
 'tape_roll_no',
 'sticker_serial_mod',
 'shelf_slot_no',
 'delivery_hours']

In [ ]:
cols = list(raw_ship_df.columns)
cols

['shipment_id',
 'hub_id',
 'service_type',
 'payment_mode',
 'parcel_weight_kg',
 'distance_km',
 'declared_value',
 'n_stops',
 'rain_flag',
 'sale_event_flag',
 'dispatch_time',
 'label_print_batch',
 'scanner_build_no',
 'tape_roll_no',
 'sticker_serial_mod',
 'shelf_slot_no',
 'delivery_hours']

In [ ]:
raw_ship_df.dtypes

,0
shipment_id,object
hub_id,object
service_type,object
payment_mode,object
parcel_weight_kg,float64
distance_km,float64
declared_value,object
n_stops,int64
rain_flag,int64
sale_event_flag,int64


In [ ]:
raw_ship_df.isna().sum()

,0
shipment_id,0
hub_id,0
service_type,0
payment_mode,18
parcel_weight_kg,25
distance_km,36
declared_value,12
n_stops,0
rain_flag,0
sale_event_flag,0


In [ ]:
raw_ship_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1224 entries, 0 to 1223
Data columns (total 17 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   shipment_id         1224 non-null   object 
 1   hub_id              1224 non-null   object 
 2   service_type        1224 non-null   object 
 3   payment_mode        1206 non-null   object 
 4   parcel_weight_kg    1199 non-null   float64
 5   distance_km         1188 non-null   float64
 6   declared_value      1212 non-null   object 
 7   n_stops             1224 non-null   int64  
 8   rain_flag           1224 non-null   int64  
 9   sale_event_flag     1224 non-null   int64  
 10  dispatch_time       1224 non-null   object 
 11  label_print_batch   1224 non-null   int64  
 12  scanner_build_no    1224 non-null   int64  
 13  tape_roll_no        1224 non-null   int64  
 14  sticker_serial_mod  1224 non-null   float64
 15  shelf_slot_no       1224 non-null   int64  
 16  delive

In [ ]:
missing_counts = raw_ship_df.isna().sum()

print(missing_counts[missing_counts > 0])

payment_mode        18
parcel_weight_kg    25
distance_km         36
declared_value      12
dtype: int64


In [ ]:
len(raw_ship_df.drop_duplicates())

1200

In [ ]:
len(raw_ship_df)

1224

In [ ]:
shipments = raw_ship_df.drop_duplicates()
shipments

,shipment_id,hub_id,service_type,payment_mode,parcel_weight_kg,distance_km,declared_value,n_stops,rain_flag,sale_event_flag,dispatch_time,label_print_batch,scanner_build_no,tape_roll_no,sticker_serial_mod,shelf_slot_no,delivery_hours
0,SHP10001,HUB10,express,COD,0.62,398.3,"₹1,290",62,0,0,2026-08-23 19:40,4594,25,167,0.650,13,34.79
1,SHP10002,HUB10,Standard,COD,4.25,312.2,"₹1,030",80,0,0,2026-08-16 07:42,4066,31,169,0.045,243,58.56
2,SHP10003,HUB21,Economy,Prepaid,0.81,421.3,₹441,73,0,0,2026-08-01 12:15,3090,25,170,0.915,80,76.56
3,SHP10004,HUB03,Standard,Prepaid,4.43,240.7,"₹3,122",42,0,0,2026-07-20 17:44,1796,21,338,0.380,175,41.18
4,SHP10005,HUB14,Standard,Prepaid,0.91,317.9,₹276,72,0,0,2026-07-19 17:31,4231,34,215,0.824,162,47.32
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1195,SHP11196,HUB19,Standard,COD,5.03,536.0,"₹1,036",90,0,0,2026-09-12 10:48,7591,36,344,0.383,192,66.44
1196,SHP11197,HUB05,Economy,Prepaid,1.65,116.2,"₹1,122",51,0,0,2026-07-21 08:39,9701,21,413,0.595,102,64.30
1197,SHP11198,HUB11,Economy,COD,1.37,165.6,₹288,76,0,0,2026-08-09 15:41,7098,31,448,0.854,97,72.10
1198,SHP11199,HUB11,Express,COD,1.71,284.6,"₹4,636",73,0,0,2026-07-25 11:37,9966,9,67,0.179,130,27.55


In [ ]:
raw_ship_df.shape

(1224, 17)

In [ ]:
shipments["service_type"].unique()

array(['express', 'Standard', 'Economy', 'Express', 'STANDARD',
       '  Express ', '  Standard ', '  Economy ', 'ECONOMY', 'EXPRESS',
       'standard', 'economy'], dtype=object)

In [ ]:
shipments["service_type"] = (shipments["service_type"].str.strip().str.title())


/tmp/ipykernel_2497/1854524958.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  shipments["service_type"] = (shipments["service_type"].str.strip().str.title())


In [ ]:
shipments["service_type"].unique()

array(['Express', 'Standard', 'Economy'], dtype=object)

In [ ]:
shipments["service_type"].value_counts()

,count
service_type,
Standard,642
Economy,315
Express,243


In [ ]:
#What is the % contribution of each service type?

In [ ]:
shipments["service_type"].value_counts()*100/len(shipments)

,count
service_type,
Standard,53.50
Economy,26.25
Express,20.25


In [ ]:
shipments["service_type"].value_counts(normalize = True)*100

,proportion
service_type,
Standard,53.50
Economy,26.25
Express,20.25


In [ ]:
shipments["payment_mode"].isna().sum()

np.int64(18)

In [ ]:
shipments["payment_mode"].value_counts()

,count
payment_mode,
Prepaid,729
COD,453


In [ ]:
len(shipments)

1200

In [ ]:
shipments["payment_mode"] = (
    shipments["payment_mode"]
    .fillna("Unknown")
)

/tmp/ipykernel_2497/3024754691.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  shipments["payment_mode"] = (


In [ ]:
shipments["payment_mode"].value_counts()

,count
payment_mode,
Prepaid,729
COD,453
Unknown,18


In [ ]:
shipments["declared_value_inr"] = (
    shipments["declared_value"]
    .str.replace("₹", "", regex=False)
    .str.replace(",", "", regex=False)
    .astype(float)
)

/tmp/ipykernel_2497/4189720518.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  shipments["declared_value_inr"] = (


In [ ]:
shipments = shipments.drop(
    columns=["declared_value"]
)

In [ ]:
shipments

,shipment_id,hub_id,service_type,payment_mode,parcel_weight_kg,distance_km,n_stops,rain_flag,sale_event_flag,dispatch_time,label_print_batch,scanner_build_no,tape_roll_no,sticker_serial_mod,shelf_slot_no,delivery_hours,declared_value_inr
0,SHP10001,HUB10,Express,COD,0.62,398.3,62,0,0,2026-08-23 19:40,4594,25,167,0.650,13,34.79,1290.0
1,SHP10002,HUB10,Standard,COD,4.25,312.2,80,0,0,2026-08-16 07:42,4066,31,169,0.045,243,58.56,1030.0
2,SHP10003,HUB21,Economy,Prepaid,0.81,421.3,73,0,0,2026-08-01 12:15,3090,25,170,0.915,80,76.56,441.0
3,SHP10004,HUB03,Standard,Prepaid,4.43,240.7,42,0,0,2026-07-20 17:44,1796,21,338,0.380,175,41.18,3122.0
4,SHP10005,HUB14,Standard,Prepaid,0.91,317.9,72,0,0,2026-07-19 17:31,4231,34,215,0.824,162,47.32,276.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1195,SHP11196,HUB19,Standard,COD,5.03,536.0,90,0,0,2026-09-12 10:48,7591,36,344,0.383,192,66.44,1036.0
1196,SHP11197,HUB05,Economy,Prepaid,1.65,116.2,51,0,0,2026-07-21 08:39,9701,21,413,0.595,102,64.30,1122.0
1197,SHP11198,HUB11,Economy,COD,1.37,165.6,76,0,0,2026-08-09 15:41,7098,31,448,0.854,97,72.10,288.0
1198,SHP11199,HUB11,Express,COD,1.71,284.6,73,0,0,2026-07-25 11:37,9966,9,67,0.179,130,27.55,4636.0


In [ ]:
shipments.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1200 entries, 0 to 1199
Data columns (total 17 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   shipment_id         1200 non-null   object 
 1   hub_id              1200 non-null   object 
 2   service_type        1200 non-null   object 
 3   payment_mode        1200 non-null   object 
 4   parcel_weight_kg    1176 non-null   float64
 5   distance_km         1164 non-null   float64
 6   n_stops             1200 non-null   int64  
 7   rain_flag           1200 non-null   int64  
 8   sale_event_flag     1200 non-null   int64  
 9   dispatch_time       1200 non-null   object 
 10  label_print_batch   1200 non-null   int64  
 11  scanner_build_no    1200 non-null   int64  
 12  tape_roll_no        1200 non-null   int64  
 13  sticker_serial_mod  1200 non-null   float64
 14  shelf_slot_no       1200 non-null   int64  
 15  delivery_hours      1200 non-null   float64
 16  declared_va

In [ ]:
shipments["dispatch_time"] = pd.to_datetime(
    shipments["dispatch_time"]
)
shipments.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1200 entries, 0 to 1199
Data columns (total 17 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   shipment_id         1200 non-null   object        
 1   hub_id              1200 non-null   object        
 2   service_type        1200 non-null   object        
 3   payment_mode        1200 non-null   object        
 4   parcel_weight_kg    1176 non-null   float64       
 5   distance_km         1164 non-null   float64       
 6   n_stops             1200 non-null   int64         
 7   rain_flag           1200 non-null   int64         
 8   sale_event_flag     1200 non-null   int64         
 9   dispatch_time       1200 non-null   datetime64[ns]
 10  label_print_batch   1200 non-null   int64         
 11  scanner_build_no    1200 non-null   int64         
 12  tape_roll_no        1200 non-null   int64         
 13  sticker_serial_mod  1200 non-null   float64       
 1

In [ ]:
sentinel_count = (
    shipments["parcel_weight_kg"] > 100
).sum()

shipments.loc[
    shipments["parcel_weight_kg"] > 100,
    "parcel_weight_kg"
] = np.nan

In [ ]:
toy_weights = pd.Series([
    1.0, 2.0, np.nan, 3.0, 4.0
])

print("Mean:", toy_weights.mean())
print("Median:", toy_weights.median())

filled = toy_weights.fillna(
    toy_weights.median()
)

print(filled)

Mean: 2.5
Median: 2.5
0    1.0
1    2.0
2    2.5
3    3.0
4    4.0
dtype: float64


In [ ]:
toy_weights = pd.Series([
    1.0, 2.0, np.nan, 3.0, 4.0
])

print("Mean:", toy_weights.mean())
print("Median:", toy_weights.median())

Mean: 2.5
Median: 2.5


In [ ]:
toy_weights = pd.Series([
    1.0, 2.0, np.nan, 3.0, 4.0, 100.0
])

print("Mean:", toy_weights.mean())
print("Median:", toy_weights.median())

Mean: 22.0
Median: 3.0


In [ ]:
#Mean is sensitive to outliers whereas median is robust (not sensitive) to outliers.

In [ ]:
medians = {
    "distance_km":
        round(float(shipments["distance_km"].median()),2),

    "parcel_weight_kg":
        round(float(shipments["parcel_weight_kg"].median()),2),

    "declared_value_inr":
        round(float(shipments["declared_value_inr"].median()),2)
}

print(medians)

{'distance_km': 314.95, 'parcel_weight_kg': 1.83, 'declared_value_inr': 1121.5}


#Sample requirements.txt file

In [ ]:
import numpy
import pandas
import sklearn
import joblib

In [ ]:
# pandas==2.2.3
# numpy==<your-installed-version>
# scikit-learn==<your-installed-version>
# joblib==<your-installed-version>


In [ ]:
print("Pandas:", pd.__version__)

Pandas: 2.2.3


In [ ]:
print("Numpy:", np.__version__)

Numpy: 2.1.3


In [ ]:
print("Scikit-learn:", sklearn.__version__)

Scikit-learn: 1.6.1


In [ ]:
print("Joblib:", joblib.__version__)

Joblib: 1.6.0


In [ ]:
times = pd.Series([
    "2026-07-01 09:30",
    "2026-07-04 18:45"
])

In [ ]:
times

,0
0,2026-07-01 09:30
1,2026-07-04 18:45


In [ ]:
times = pd.to_datetime(times)

print(times)
print(times.dt.hour)

0   2026-07-01 09:30:00
1   2026-07-04 18:45:00
dtype: datetime64[ns]
0     9
1    18
dtype: int32


In [ ]:
times.dt.dayofweek

,0
0,2
1,5


In [ ]:
# 0 --> Monday
# 1 --> Tue

In [ ]:
weights = pd.Series([
    1.5, 2.0, 4.2, 999.0, 3.1
])

#greater than 100 --> set to nan
weights[weights > 100] = np.nan

In [ ]:
weights

,0
0,1.5
1,2.0
2,4.2
3,NaN
4,3.1


In [ ]:
medians

{'distance_km': 314.95, 'parcel_weight_kg': 1.83, 'declared_value_inr': 1121.5}

In [ ]:
for k in medians.keys():
  print(k)

distance_km
parcel_weight_kg
declared_value_inr


In [ ]:
for v in medians.values():
  print(v)

314.95
1.83
1121.5


In [ ]:
for i in medians.items():
  print(i)

('distance_km', 314.95)
('parcel_weight_kg', 1.83)
('declared_value_inr', 1121.5)


In [ ]:
for i in medians.items():
  print("Key", i[0], ", value:", i[1])

Key distance_km , value: 314.95
Key parcel_weight_kg , value: 1.83
Key declared_value_inr , value: 1121.5


In [ ]:
for i in medians.items():
  k,v = i
  print(f"key : {k}, value : {v}")

key : distance_km, value : 314.95
key : parcel_weight_kg, value : 1.83
key : declared_value_inr, value : 1121.5


In [ ]:
#double mentos zindagi

for k,v in medians.items():
  print(f"key : {k}, value : {v}")

key : distance_km, value : 314.95
key : parcel_weight_kg, value : 1.83
key : declared_value_inr, value : 1121.5


In [ ]:
#replace respective column's missing values with their medians

for column, median_value in medians.items():
    shipments[column] = (
        shipments[column].fillna(median_value)
    )

In [ ]:
  shipments[
        [
            "distance_km",
            "parcel_weight_kg",
            "declared_value_inr"
        ]
    ].isna().sum()

,0
distance_km,0
parcel_weight_kg,0
declared_value_inr,0


#Feature Engineering

In [ ]:
dispatch_time = 2026-07-04 19:30
payment_mode = COD
service_type = Express
declared_value_inr = 1290

In [ ]:
# Engineered Features:
# 1- dispatch_hour --> 19
# 2- is_weekend --> 1
# 3--> is_cod --> 1
# 4 --> promised_hrs = 36

In [ ]:
# break --> 12:43

In [ ]:
hubs_df

,hub_id,hub_city,region,hub_tier,hub_daily_capacity
0,HUB01,Mumbai,West,1,1500
1,HUB02,Delhi,North,1,1600
2,HUB03,Bengaluru,South,1,1450
3,HUB04,Hyderabad,South,1,1350
4,HUB05,Chennai,South,1,1300
5,HUB06,Pune,West,1,1250
6,HUB07,Kolkata,East,1,1200
7,HUB08,Ahmedabad,West,1,1150
8,HUB09,Jaipur,North,2,950
9,HUB10,Lucknow,North,2,900


In [ ]:
hubs_df.head()

,hub_id,hub_city,region,hub_tier,hub_daily_capacity
0,HUB01,Mumbai,West,1,1500
1,HUB02,Delhi,North,1,1600
2,HUB03,Bengaluru,South,1,1450
3,HUB04,Hyderabad,South,1,1350
4,HUB05,Chennai,South,1,1300


In [ ]:
toy_shipments = pd.DataFrame({
    "shipment": ["A", "B", "C"],
    "hub_id": ["HUB01", "HUB02", "HUB01"]
})

toy_hubs = pd.DataFrame({
    "hub_id": ["HUB01", "HUB02"],
    "hub_city": ["Mumbai", "Delhi"]
})


In [ ]:
toy_shipments

,shipment,hub_id
0,A,HUB01
1,B,HUB02
2,C,HUB01


In [ ]:
toy_hubs

,hub_id,hub_city
0,HUB01,Mumbai
1,HUB02,Delhi


In [ ]:
merged = toy_shipments.merge(
    toy_hubs,
    on="hub_id",
    how="left"
)


In [ ]:
merged

,shipment,hub_id,hub_city
0,A,HUB01,Mumbai
1,B,HUB02,Delhi
2,C,HUB01,Mumbai


In [ ]:
PROMISED_HOURS = {
    "Express": 36,
    "Standard": 60,
    "Economy": 96
}

FEATURES = [
    "promised_hours",
    "distance_km",
    "parcel_weight_kg",
    "log_declared_value",
    "n_stops",
    "rain_flag",
    "sale_event_flag",
    "is_cod",
    "is_weekend",
    "dispatch_hour",
    "hub_tier",
    "hub_daily_capacity",
    "label_print_batch",
    "scanner_build_no",
    "tape_roll_no",
    "sticker_serial_mod",
    "shelf_slot_no"
]

NOISE_COLUMNS = [
    "label_print_batch",
    "scanner_build_no",
    "tape_roll_no",
    "sticker_serial_mod",
    "shelf_slot_no"
]

In [ ]:
 result = shipments.merge(
        hubs[
            [
                "hub_id",
                "hub_city",
                "region",
                "hub_tier",
                "hub_daily_capacity"
            ]
        ],
        on="hub_id",
        how="left",
        validate="many_to_one"
    )


In [ ]:
result.head()

,shipment_id,hub_id,service_type,payment_mode,parcel_weight_kg,distance_km,n_stops,rain_flag,sale_event_flag,dispatch_time,...,scanner_build_no,tape_roll_no,sticker_serial_mod,shelf_slot_no,delivery_hours,declared_value_inr,hub_city,region,hub_tier,hub_daily_capacity
0,SHP10001,HUB10,Express,COD,0.62,398.3,62,0,0,2026-08-23 19:40:00,...,25,167,0.650,13,34.79,1290.0,Lucknow,North,2,900
1,SHP10002,HUB10,Standard,COD,4.25,312.2,80,0,0,2026-08-16 07:42:00,...,31,169,0.045,243,58.56,1030.0,Lucknow,North,2,900
2,SHP10003,HUB21,Economy,Prepaid,0.81,421.3,73,0,0,2026-08-01 12:15:00,...,25,170,0.915,80,76.56,441.0,Bhubaneswar,East,3,390
3,SHP10004,HUB03,Standard,Prepaid,4.43,240.7,42,0,0,2026-07-20 17:44:00,...,21,338,0.380,175,41.18,3122.0,Bengaluru,South,1,1450
4,SHP10005,HUB14,Standard,Prepaid,0.91,317.9,72,0,0,2026-07-19 17:31:00,...,34,215,0.824,162,47.32,276.0,Coimbatore,South,2,740


In [ ]:
  result["promised_hours"] = (
        result["service_type"]
        .map(PROMISED_HOURS)
        .astype(int)
    )

In [ ]:
result

,shipment_id,hub_id,service_type,payment_mode,parcel_weight_kg,distance_km,n_stops,rain_flag,sale_event_flag,dispatch_time,...,tape_roll_no,sticker_serial_mod,shelf_slot_no,delivery_hours,declared_value_inr,hub_city,region,hub_tier,hub_daily_capacity,promised_hours
0,SHP10001,HUB10,Express,COD,0.62,398.3,62,0,0,2026-08-23 19:40:00,...,167,0.650,13,34.79,1290.0,Lucknow,North,2,900,36
1,SHP10002,HUB10,Standard,COD,4.25,312.2,80,0,0,2026-08-16 07:42:00,...,169,0.045,243,58.56,1030.0,Lucknow,North,2,900,60
2,SHP10003,HUB21,Economy,Prepaid,0.81,421.3,73,0,0,2026-08-01 12:15:00,...,170,0.915,80,76.56,441.0,Bhubaneswar,East,3,390,96
3,SHP10004,HUB03,Standard,Prepaid,4.43,240.7,42,0,0,2026-07-20 17:44:00,...,338,0.380,175,41.18,3122.0,Bengaluru,South,1,1450,60
4,SHP10005,HUB14,Standard,Prepaid,0.91,317.9,72,0,0,2026-07-19 17:31:00,...,215,0.824,162,47.32,276.0,Coimbatore,South,2,740,60
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1195,SHP11196,HUB19,Standard,COD,5.03,536.0,90,0,0,2026-09-12 10:48:00,...,344,0.383,192,66.44,1036.0,Guwahati,East,3,450,60
1196,SHP11197,HUB05,Economy,Prepaid,1.65,116.2,51,0,0,2026-07-21 08:39:00,...,413,0.595,102,64.30,1122.0,Chennai,South,1,1300,96
1197,SHP11198,HUB11,Economy,COD,1.37,165.6,76,0,0,2026-08-09 15:41:00,...,448,0.854,97,72.10,288.0,Chandigarh,North,2,850,96
1198,SHP11199,HUB11,Express,COD,1.71,284.6,73,0,0,2026-07-25 11:37:00,...,67,0.179,130,27.55,4636.0,Chandigarh,North,2,850,36


In [ ]:
    result["is_cod"] = (
        result["payment_mode"] == "COD"
    ).astype(int)

In [ ]:
   result["is_weekend"] = (
        result["dispatch_time"].dt.dayofweek >= 5
    ).astype(int)

In [ ]:
   result["dispatch_hour"] = (
        result["dispatch_time"].dt.hour
    ).astype(int)